<a href="https://colab.research.google.com/github/souvikdey214-code/Krutanic_Weekly-Project-Task/blob/main/09_ChatForge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math, random
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import GPT2Tokenizer, GPT2LMHeadModel

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

base_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)


ids = tokenizer.encode("Hello, how are you", return_tensors="pt").to(device)
out = base_model.generate(ids, max_new_tokens=30, do_sample=True, top_p=0.9,
                          pad_token_id=tokenizer.eos_token_id)
print("\nBase GPT-2 says:", tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
from datasets import load_dataset

N_DIALOGS = 5000
MAX_LEN   = 256

raw, field = None, None
for ds_id in ["roskoN/dailydialog", "li2017dailydialog/daily_dialog", "daily_dialog"]:
    try:
        raw = load_dataset(ds_id, split="train")
        field = "utterances" if "utterances" in raw.column_names else "dialog"
        print(f"Loaded '{ds_id}'  (dialog field: '{field}')")
        break
    except Exception as e:
        print(f"Could not load {ds_id}: {type(e).__name__}")
assert raw is not None, "No DailyDialog source worked - search the HF Hub for 'dailydialog' and paste its id above."

raw = raw.shuffle(seed=SEED).select(range(min(N_DIALOGS, len(raw))))

def format_dialog(example):
    turns = example[field]
    text = "".join(f"SPEAKER_{'A' if i % 2 == 0 else 'B'}: {t.strip()}\n" for i, t in enumerate(turns))
    return {"text": text + tokenizer.eos_token}

raw = raw.map(format_dialog)
print("\nExample:\n", raw[0]["text"][:400])

split = raw.train_test_split(test_size=0.1, seed=SEED)
tok = lambda ex: tokenizer(ex["text"], truncation=True, max_length=MAX_LEN)
train_ds = split["train"].map(tok, batched=True, remove_columns=split["train"].column_names)
val_ds   = split["test"].map(tok, batched=True, remove_columns=split["test"].column_names)
print(f"\nTrain: {len(train_ds)} dialogs | Validation: {len(val_ds)} dialogs")

In [ ]:
from transformers import (Trainer, TrainingArguments, DataCollatorForLanguageModeling,
                          EarlyStoppingCallback)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args_kwargs = dict(
    output_dir="chatforge_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)
try:
    args = TrainingArguments(eval_strategy="steps", save_strategy="steps", **args_kwargs)
except TypeError:
    args = TrainingArguments(evaluation_strategy="steps", save_strategy="steps", **args_kwargs)

model = GPT2LMHeadModel.from_pretrained("gpt2")
trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [ ]:
trainer.train()
trainer.save_model("chatforge_model")
tokenizer.save_pretrained("chatforge_model")
print("Saved to ./chatforge_model")

In [ ]:
log = trainer.state.log_history
tr = [(l["step"], l["loss"]) for l in log if "loss" in l]
ev = [(l["step"], l["eval_loss"]) for l in log if "eval_loss" in l]
plt.figure(figsize=(8, 4))
if tr: plt.plot(*zip(*tr), label="train loss")
if ev: plt.plot(*zip(*ev), "o-", label="eval loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=.3); plt.title("Fine-tuning curves")
plt.show()

In [ ]:
model = trainer.model
model.eval()

def generate_reply(prompt, history="", max_length=100, temperature=0.7, top_p=0.9, top_k=50):
    """Return the model's SPEAKER_B reply to `prompt`, given earlier turns in `history`."""
    text = f"{history}SPEAKER_A: {prompt}\nSPEAKER_B:"
    ids = tokenizer.encode(text, return_tensors="pt")
    ids = ids[:, -(1024 - max_length):].to(model.device)
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=max_length, do_sample=True,
            temperature=temperature, top_p=top_p, top_k=top_k,
            pad_token_id=tokenizer.eos_token_id,
        )
    reply = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    return reply.split("\n")[0].split("SPEAKER_A:")[0].strip()

test_prompts = [
    "Hi, how are you today?", "What do you do for fun?", "Can you recommend a good movie?",
    "I'm feeling a bit tired.", "What's your favourite food?", "Do you like travelling?",
    "How was your weekend?", "Could you help me with my homework?", "What time is it?", "Goodbye!",
]
for p in test_prompts:
    print(f"You: {p}\nBot: {generate_reply(p)}\n")

In [ ]:
history = ""
MAX_HISTORY_CHARS = 1500       # simple context-window management

print("ChatForge ready. Type 'quit' to exit.\n")
while True:
    user_input = input("You: ").strip()
    if user_input.lower() in {"quit", "exit"}:
        print("Bot: Goodbye!")
        break
    if not user_input:
        continue
    reply = generate_reply(user_input, history)
    print(f"Bot: {reply}\n")
    history += f"SPEAKER_A: {user_input}\nSPEAKER_B: {reply}\n"
    history = history[-MAX_HISTORY_CHARS:]

ChatForge ready. Type 'quit' to exit.



KeyboardInterrupt: Interrupted by user

In [ ]:
import gradio as gr

def to_history_text(history):
    """Handles both Gradio history formats (list of pairs, or list of role/content dicts)."""
    lines = []
    for h in history:
        if isinstance(h, dict):
            who = "A" if h["role"] == "user" else "B"
            lines.append(f"SPEAKER_{who}: {h['content']}\n")
        else:
            user, bot = h
            lines.append(f"SPEAKER_A: {user}\n")
            if bot: lines.append(f"SPEAKER_B: {bot}\n")
    return "".join(lines)[-MAX_HISTORY_CHARS:]

def gradio_chat(message, history):
    return generate_reply(message, to_history_text(history))

demo = gr.ChatInterface(gradio_chat, title="ChatForge", description="GPT-2 fine-tuned on DailyDialog")
demo.launch(share=True)

In [ ]:
base_trainer = Trainer(model=base_model, args=args, eval_dataset=val_ds, data_collator=collator)

ppl_base  = math.exp(base_trainer.evaluate()["eval_loss"])
ppl_tuned = math.exp(trainer.evaluate()["eval_loss"])
print(f"Perplexity  base GPT-2 : {ppl_base:8.2f}")
print(f"Perplexity  fine-tuned : {ppl_tuned:8.2f}   ({(1 - ppl_tuned / ppl_base):.0%} lower)")

In [ ]:
all_utterances = [u.strip() for d in raw[field][:500] for u in d]
prompts = random.sample(test_prompts, 5)
key = []
for i, p in enumerate(prompts, 1):
    bot, rnd = generate_reply(p), random.choice(all_utterances)
    pair = [("chatbot", bot), ("random", rnd)]
    random.shuffle(pair)
    key.append((i, [name for name, _ in pair]))
    print(f"Q{i}: {p}\n   (1) {pair[0][1]}\n   (2) {pair[1][1]}\n")
print("ANSWER KEY (hide until the test is done):")
for i, order in key:
    print(f"  Q{i}: (1)={order[0]}, (2)={order[1]}")